In [1]:
from utils import * 
import numpy as np
import shutil 
from matplotlib.lines import Line2D
import itertools
import os
import re
import sys 

%load_ext autoreload 
%autoreload 2

# TODO
# (1) Check if all of the AlphaFold databases are available on biotite. 


# Custom MSAs

Analyzing the results of ColabFold structure prediction makes it clear that many of the Betazoid proteins have none (or very few) proteins populating their MSAs, and consequently have very weak folds. 

According to the [AlphaFold3 documentation](https://github.com/google-deepmind/alphafold3/blob/main/docs/installation.md), AlphaFold3 requires at least 3 of the following databases.
1. BFD
2.  MGnify
3. PDB 
4. UniProt
5. UniRef90

Although AlphaFold's JackHMMer-based homology search is unable to recover homologs for the Betazoid genes in these databases, there are likely far more homologs in ggKbase (particulary for the genes we already know are conserved amongst Betazoids). Here, we construct custom MSAs for select Betazoid proteins using ggKbase BLAST hits. 



In [2]:
level_1_conserved_cluster_ids = [8, 7, 0, 4, 5, 6, 1, 3, 2] # Gene clusters found in > 7 Betazoids. 
level_2_conserved_cluster_ids = [9, 12, 10, 13, 11] # Gene clusters found in > 5 Betazoids
level_3_conserved_cluster_ids = [18, 42, 33, 25, 17, 19, 20, 21, 50, 31, 40, 27, 34, 15, 14, 54, 24, 23, 41, 39, 57, 47, 48, 51, 45, 35, 49, 28, 62, 29] # Gene clusters found in > 1 Betazoid.

conserved_cluster_ids = level_1_conserved_cluster_ids + level_2_conserved_cluster_ids + level_3_conserved_cluster_ids

genes_df = pd.read_csv('../data/genes/genes.csv', index_col=0)
genes_df['cluster_id'] = genes_df.index.map(json.load(open('../data/genes/clusters.json', 'r')))
genes_df = genes_df[genes_df.cluster_id.isin(conserved_cluster_ids)].copy()

print('Number of conserved gene clusters:', len(conserved_cluster_ids))
print('Number of conserved genes:', len(genes_df))


Number of conserved gene clusters: 44
Number of conserved genes: 196


## BLASTp search against ggKbase

The ggKbase web iterface only allows one query sequence at a time. Even limiting the queries to genes in the conserved gene clusters, searching each individually would require nearly individual queries. 

*Instead of using the web-based, I asked Shufei to run the BLAST queries for me, and provide the corresponding amino acid sequences.I used lenient search parameters, with a maximum allowed E-value of 10.*

In [3]:
FASTAFile.from_df(genes_df).write('../data/genes/conserved_clusters.faa') # Write the query sequences to a FASTA file. 

In [4]:
ggkbase_dir = '../data/genes/blast/ggkbase' # Directory where the BLAST search outputs are stored. 

In [ ]:


ggkbase_df = BLASTFile.from_file(os.path.join(GGKBASE_DIR, 'level_1_conserved_clusters.txt')).to_df()
ggkbase_df['query_coverage'] = ggkbase_df.alignment_length / ggkbase_df.query_length
ggkbase_df['target_coverage'] = ggkbase_df.alignment_length / ggkbase_df.target_length
ggkbase_df['query_cluster_id'] = ggkbase_df.query_id.map(genes_df.cluster_id)

print('Number of unique hits across all clusters before filtering:', ggkbase_df.target_id.nunique())

filters = dict()
filters['low_query_coverage'] = ggkbase_df.query_coverage < 0.75
filters['low_target_coverage'] = ggkbase_df.target_coverage < 0.75
filters['different_lengths'] = (np.abs(ggkbase_df.target_length - ggkbase_df.query_length) / ggkbase_df.query_length) > 0.5
filters['low_percent_identity'] = ggkbase_df.percent_identity < 0.2 # Probably pushing what would make sense for an MSA. 

ggkbase_df = apply_filters(filters, ggkbase_df)

seqs = FASTAFile.from_file(os.path.join(GGKBASE_DIR, 'level_1_conserved_clusters.faa')).to_df().seq
seqs = seqs.str.replace('*', '', regex=False) # Remove the '*' characters. 
seqs.index = seqs.index.str.split('|').str[0] # Remove the bin and project ID from the header. 
seqs = seqs.to_dict()

ggkbase_df['seq'] = ggkbase_df.target_id.map(seqs)
assert not np.any(ggkbase_df.seq.isnull()), f'{ggkbase_df.seq.isnull().sum()} out of {len(ggkbase_df)} ggKbase genes do not have a sequence.'

print('Number of unique sequences IDs across all clusters after filtering:', ggkbase_df.seq.nunique())
mask = ggkbase_df.seq.duplicated(keep='first') | ggkbase_df.target_id.duplicated(keep='first')
assert mask.sum() == ggkbase_df.seq.duplicated(keep='first').sum() , 'Not all entries with the same ID have the same sequence.'
ggkbase_df = ggkbase_df[~mask].copy().set_index('target_id')

for cluster_id, df in ggkbase_df.groupby('query_cluster_id'):
    print(f'\nCluster {int(cluster_id)} ({len(df)} unique hits)')
    print(f'Minimum percent identity: {100 * df.percent_identity.min():.2f}%')
    print(f'Minimum bit score: {df.bit_score.min()}')
    print(f'Maximum E-value: {df.e_value.max()}')

    # Write the cluster hits to a FASTA file. 
    path = os.path.join(GGKBASE_DIR, f'cluster_{int(cluster_id)}.faa')
    FASTAFile.from_df(df).write(path)
